# CS 3120/5120: Secure Distributed Computation
## Homework 8

In [ ]:
# Useful imports and utility functions
import pychor
import numpy as np
from numpy.polynomial import polynomial as poly
from dataclasses import dataclass
from typing import Dict, Tuple, List

## Code for the FV12 FHE scheme

In [ ]:
from numpy.polynomial import polynomial as poly

# Constants for the system
n = 2**0
size = n
q = 2**15
modulus = q
t = 2
p = q**3
std1 = 0
std2 = 0

poly_mod = np.array([1] + [0] * (n - 1) + [1])

# Add and multiply polynomials, *without* modulus
def polyadd_wm(x, y, poly_mod):
    return poly.polydiv(poly.polyadd(x, y), poly_mod)[1]

def polymul_wm(x, y, poly_mod):
    return poly.polydiv(poly.polymul(x, y), poly_mod)[1]

# Add and multiply polynomials, *with* modulus
def polyadd(x, y, modulus, poly_mod):
    return np.int64(np.round(poly.polydiv(poly.polyadd(x, y) % modulus, poly_mod)[1] % modulus))

def polymul(x, y, modulus, poly_mod):
    return np.int64(np.round(poly.polydiv(poly.polymul(x, y) % modulus, poly_mod)[1] % modulus))

# Functions to generate important polynomials
def gen_binary_poly(size):
    return np.random.randint(0, 2, size, dtype=np.int64)

def gen_uniform_poly(size, modulus):
    return np.random.randint(0, modulus, size, dtype=np.int64)

def gen_normal_poly(size, mean, std):
    return np.int64(np.random.normal(mean, std, size=size))

def keygen():
    s = gen_binary_poly(size)
    a = gen_uniform_poly(size, modulus)
    e = gen_normal_poly(size, 0, std1)
    b = -polyadd(polymul(a, s, q, poly_mod),
                 e, q, poly_mod)
    return s, (b, a)

def encrypt(pk, m):
    assert len(m) == n
    m = np.array(m)
    delta = q // t
    u = gen_binary_poly(size)
    e1 = gen_normal_poly(size, 0, std1)
    e2 = gen_normal_poly(size, 0, std1)

    # p0 · u + e1 + ∆ · m
    ct0_1 = polymul(pk[0], u, q, poly_mod)
    ct0_2 = e1
    ct0_3 = delta * m
    ct0 = polyadd(ct0_1, polyadd(ct0_2, ct0_3, q, poly_mod),
                  q, poly_mod)
    # p1 · u + e2
    ct1 = polyadd(polymul(pk[1], u, q, poly_mod),
                  e2, q, poly_mod)
    return ct0, ct1

def pad(poly, size):
    padding_size = size - len(poly)
    zeros = [0] * padding_size
    return np.concatenate((poly, zeros)).astype(int)

def decrypt(sk, ct):
    # c0 + c1 · s
    scaled_pt = polyadd(ct[0], polymul(ct[1], sk, q, poly_mod), q, poly_mod)
    #print('Un-rounded decrypted value:', t*scaled_pt/ q)
    decrypted_poly = np.round(t * scaled_pt / q) % t
    return pad(decrypted_poly, size)

def e_add(ct1, ct2):
    # just add the ciphertexts
    new_ct0 = polyadd(ct1[0], ct2[0], q, poly_mod)
    new_ct1 = polyadd(ct1[1], ct2[1], q, poly_mod)
    return new_ct0, new_ct1

def eval_keygen(sk):
    # [−(a · s + e) + p · s2]p·q , a
    a = gen_uniform_poly(size, p*q)
    e = gen_normal_poly(size, 0, std2)
    psk = p * poly.polymul(sk, sk) # warning: weird

    b = polyadd_wm(-polyadd_wm(polymul_wm(a, sk, poly_mod),
                               e, poly_mod),
                   psk, poly_mod) % (p*q)

    return b, a

def e_mul(ct1, ct2, rlk):
    # step 1: multiply ciphertexts
    c0 = np.int64(np.round(polymul_wm(ct1[0], ct2[0], poly_mod) * t / q)) % q
    c1 = np.int64(np.round(polyadd_wm(polymul_wm(ct1[0], ct2[1], poly_mod),
                                      polymul_wm(ct1[1], ct2[0], poly_mod), poly_mod) * t / q)) % q
    c2 = np.int64(np.round(polymul_wm(ct1[1], ct2[1], poly_mod) * t / q)) % q

    # step 2: relinearization
    c_20 = np.int64(np.round(polymul_wm(c2, rlk[0], poly_mod) / p)) % q
    c_21 = np.int64(np.round(polymul_wm(c2, rlk[1], poly_mod) / p)) % q

    new_c0 = polyadd(c0, c_20, q, poly_mod)
    new_c1 = polyadd(c1, c_21, q, poly_mod)
    return new_c0, new_c1

## Question 1 (30 points)

Implement the `FHEBit` class, a class that encapsulates operations on a bit encrypted using the FV scheme. It should support addition and multiplication. See the test case for an example. Reference the `SecBit` class implementation from [the GMW protocol in chapter 7](https://jnear.github.io/programming-mpc/chapters/chapter07.html#id2) (but note that there is no need for communication in this case).

In [ ]:
@dataclass
class FHEBit:
    bit: Tuple[np.poly, np.poly]   # holds the encrypted bit
    pk:  Tuple[np.poly, np.poly]   # holds the public key
    rlk: Tuple[np.poly, np.poly]   # holds the relinearization key (needed for multiplication)
    
    @classmethod
    def input(cls, val, pk, rlk):
        """encrypt the bit `val` and return a FHEBit"""
        # YOUR CODE HERE
        raise NotImplementedError()
        
    def __add__(x, y):
        """add two FHEBits `x` and `y` and return a FHEBit"""
        # YOUR CODE HERE
        raise NotImplementedError()
        
    def __mul__(x, y):
        """multiply two FHEBits `x` and `y` and return a FHEBit"""
        # YOUR CODE HERE
        raise NotImplementedError()
        
    def reveal(self, sk):
        """decrypt the FHEBit using the secret key and return the bit"""
        # YOUR CODE HERE
        raise NotImplementedError()

In [ ]:
sk, pk = keygen()
rlk = eval_keygen(sk)

x = FHEBit.input(0, pk, rlk)
y = FHEBit.input(1, pk, rlk)

r1 = (x+y).reveal(sk)
print('x+y:', r1)
r2 = (x*y).reveal(sk)
print('x*y:', r2)
r3 = ((x+y)*y).reveal(sk)
print('(x+y)*y:', r3)

z = y
for _ in range(10):
    z = z * y
r4 = z.reveal(sk)
print('y^10:', r4)

assert r1 == 1
assert r2 == 0
assert r3 == 1
assert r4 == 1

## Question 2 (30 points)

Implement the `FHEInt` class, a class that encapsulates operations on a **4-bit integer** encrypted using the FV scheme, by representing the integer as a list of `FHEBit` objects. It should support addition and multiplication using adder and multiplier circuits. Reference the `SecBitInt` class from [Chapter 8](https://jnear.github.io/programming-mpc/chapters/chapter08.html#secbitint-secure-binary-integers), which shows how to implement integer operations in terms of bit operations.

In [ ]:
@dataclass
class FHEInt:
    bits: List[FHEBit]   # holds the encrypted bit
    
    @classmethod
    def input(cls, val, pk, rlk):
        """encrypt the integer `val` and return a FHEInt"""
        # YOUR CODE HERE
        raise NotImplementedError()
        
    def __add__(x, y):
        """add two FHEBits `x` and `y` and return a FHEBit"""
        # YOUR CODE HERE
        raise NotImplementedError()
        
    def __mul__(x, y):
        """multiply two FHEBits `x` and `y` and return a FHEBit"""
        # YOUR CODE HERE
        raise NotImplementedError()
        
    def reveal(self, sk):
        """decrypt the FHEBit using the secret key and return the bit"""
        # YOUR CODE HERE
        raise NotImplementedError()

In [ ]:
sk, pk = keygen()
rlk = eval_keygen(sk)

x = FHEInt.input(3, pk, rlk)
y = FHEInt.input(5, pk, rlk)

assert x.reveal(sk) == 3
assert y.reveal(sk) == 5

r1 = (x+y).reveal(sk)
print('x+y:', r1)

r2 = (x*y).reveal(sk)
print('x*y:', r2)

mult_results = np.array([(x*y).reveal(sk) for _ in range(20)])
print('Fraction of correct results:', np.sum(mult_results == 15) / 20)

assert r1 == 8
assert np.any(mult_results == 15)

## Question 3 (5 points)

Does addition with `FHEInt` always give the right answer? What about multiplication? If not, why not, and what parameters would you change to solve the problem?

YOUR ANSWER HERE

## Question 4 (5 points)

Try changing `n` in the first cell to a larger number (like `n=2**10`) and re-run your `FHEInt` multiplication test. How does the running time compare to the case where `n=1`? What does this mean for how fast FHE operations are likely to run in a practical application?

YOUR ANSWER HERE